In [4]:
# clone the repo - not relevant if we're running in local - arcgis arcpy
# !git clone https://github.com/miqbalf/GEE_notebook_Forestry.git temp_dir
# !mv temp_dir/* .  # Move the contents to the current directory
# !rm -rf temp_dir  # Clean up the temporary directory

In [5]:
# install library python needed
# !pip install -r requirements.txt

In [6]:
# # do it one time, better do it in terminal, for this to work with arcgis arcpy
# !conda install conda-forge::earthengine-api
# !conda install conda-forge::geemap

In [7]:
### PLOT CREATION - library and arcpy setup
import arcpy

map_name_arcgis_pro = 'sra'

# Reference the current ArcGIS Pro project
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.activeMap
map.name

'sra'

In [8]:
## PLOT CREATION - preparation input
import os

project_name = 'sra_sulawesi'

#### data path setup
current_dir = r'C:\Users\q_bal\Documents\GEE_notebook_Forestry\osi\00_input'

main_dir = current_dir
dynamic_baseline_dir = os.path.join(main_dir,'dynamic_baseline')
os.makedirs(dynamic_baseline_dir,exist_ok=True) # create folder if not exist

# copy paste the landcover to that folder (dynamic_baseline_dir) and do this
landcover = os.path.join(dynamic_baseline_dir,
                          'mof_lc_2022_aoi.shp')
input_gdb = os.path.join(dynamic_baseline_dir, 'input_fc.gdb')
scratch_gdb =  os.path.join(dynamic_baseline_dir, 'scratch_result.gdb')

In [9]:
## PLOT CREATION
# GRID DATA CREATION - SAMPLE AND CONTROL PLOTS PREPARATION BASED ON THE EXISTING LANDCOVER DATASET
arcpy.management.CreateSpatialSamplingLocations(
    in_study_area=landcover,
    out_features=os.path.join(scratch_gdb, f'grid_points_{project_name}'),
    sampling_method="SYSTEMATIC",
    strata_id_field=None,
    strata_count_method="EQUAL",
    bin_shape="SQUARE",
    bin_size=15000,
    h3_resolution=7,
    num_samples=100,
    num_samples_per_strata=100,
    population_field=None,
    geometry_type="POINT",
    min_distance="0 Meters",
    spatial_relationship="HAVE_THEIR_CENTER_IN"
)


<Result 'C:\\Users\\q_bal\\Documents\\GEE_notebook_Forestry\\osi\\00_input\\dynamic_baseline\\scratch_result.gdb\\grid_points_sra_sulawesi'>

In [1]:
import sys
import os
import json

module_path = r'C:\Users\q_bal\Documents\GEE_notebook_Forestry'
create_training_gee = False

# Add the module path to sys.path
if module_path not in sys.path:
    sys.path.append(module_path)

print('current_dir: ',current_dir)

save_dir_output = current_dir
input_dir = os.path.join(save_dir_output, '00_input')
output_dir = os.path.join(save_dir_output, '01_output')

os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

########### VARIABLE DEFINITION
is_first_run = False

dict_config = {
    "project_name" : project_name,
    "date_analyzed": '20250917',
    "save_dir_output":save_dir_output,
    "input_dir": input_dir,
    "output_dir": output_dir,
    "I_satellite":"Landsat",
    "pca_scaling":1,
    "tileScale" :1,
    "AOI_path": os.path.join(input_dir, "aoi_sra.shp"),
    "OID_field_name": "FID",
    "input_training": f"{input_dir}/training_merged_2023June_updated_2024_dec_updated.shp", # updated to 202412
    "algo_ml_selected": "gbm",
    "date_start_end":["2024-12-1","2024-12-31"],
    "super_pixel_size": 3,

    "region": "asia",

    "pixel_number": 6, #> 0.5 ha
    "year_start_loss": 14, # start 2014  (10 years rule)
    "tree_cover_forest": 10, # > 10 percent

    "band_name_image": "Class",
    "cloud_cover_threshold" : 40,
    "crs_input" : "EPSG:4326",
    "IsThermal": False,

    "fcd_selected": 21,
    "high_forest" : 65,
    "yrf_forest" : 55, # synthetic number
    "shrub_grass" : 35,
    "open_land" : 30,
    "ndwi_hi_sentinel" : 0.05,
    "ndwi_hi_landsat" : 0.1,
    "ndwi_hi_planet" : -0.2,

    "create_training_gee" : False, # if True here then following needs to fill in
    #"num_training_to_create": 8, # open, grass, shrub, crop, forest, infra, water, wetland #not relevant here
    #"location_sample_created": os.path.join(input_dir , f"new_{project_name}_training_{date_analyzed}.shp")
    }

create_training_gee = dict_config['create_training_gee']

# config_json creation - reupdate if any changes happen
config_json_loc = os.path.join(input_dir,f"{dict_config['date_analyzed']}_{dict_config['project_name']}_conf.json")
with open(config_json_loc, 'w') as json_file:
    json.dump(dict_config, json_file, indent=4)

if dict_config['I_satellite'] == 'Planet':
    Scale = 5
elif dict_config['I_satellite'] == 'Landsat':
    Scale = 30
elif dict_config['I_satellite'] == 'Sentinel':
    Scale = 10

print(f"data created in {config_json_loc}")

# using the dict config
config = dict_config

import pprint

pprint.pprint(config)


current_dir:  C:\Users\q_bal\Documents\GEE_notebook_Forestry\osi
data created in C:\Users\q_bal\Documents\GEE_notebook_Forestry\osi\00_input\20250917_sra_sulawesi_conf.json
{'AOI_path': 'C:\\Users\\q_bal\\Documents\\GEE_notebook_Forestry\\osi\\00_input\\aoi_sra.shp',
 'I_satellite': 'Landsat',
 'IsThermal': False,
 'OID_field_name': 'FID',
 'algo_ml_selected': 'gbm',
 'band_name_image': 'Class',
 'cloud_cover_threshold': 40,
 'create_training_gee': False,
 'crs_input': 'EPSG:4326',
 'date_analyzed': '20250917',
 'date_start_end': ['2024-12-1', '2024-12-31'],
 'fcd_selected': 21,
 'high_forest': 65,
 'input_dir': 'C:\\Users\\q_bal\\Documents\\GEE_notebook_Forestry\\osi\\00_input',
 'input_training': 'C:\\Users\\q_bal\\Documents\\GEE_notebook_Forestry\\osi\\00_input/training_merged_2023June_updated_2024_dec_updated.shp',
 'ndwi_hi_landsat': 0.1,
 'ndwi_hi_planet': -0.2,
 'ndwi_hi_sentinel': 0.05,
 'open_land': 30,
 'output_dir': 'C:\\Users\\q_bal\\Documents\\GEE_notebook_Forestry\\osi\\0

In [2]:
# library import
# import main library
import ee
import geemap
import osi
import pandas as pd
import geopandas as gpd
import os
import json
import arcpy

from osi import root_osi_folder
from osi.utils.main import validate_aoi
# convert the modules for image collection (cloudless masking, compositing, reducer etc)
from osi.image_collection.main import ImageCollection
from osi.spectral_indices.spectral_analysis import SpectralAnalysis
from osi.spectral_indices.utils import normalization_100
from osi.hansen.historical_loss import HansenHistorical
from osi.classifying.assign_zone import AssignClassZone
from osi.legends.utils import convert_to_legend_items
from osi.legends.main import LegendsBuilder
from osi.obia.main import OBIASegmentation
from osi.ml.main import LandcoverML
#from osi.arcpy.main import ArcpyOps  # only if using arcgis
from osi.fcd.main_fcd import FCDCalc
from osi.pca.pca_gee import PCA
from osi.hansen.historical_loss import HansenHistorical
from osi.classifying.assign_zone import AssignClassZone

arcpy.env.overwriteOutput = True

In [3]:
# project gcp
name_project_gcp = 'ee-iwansetiawan'

# Trigger the authentication flow. if you want to user json, please comment this
ee.Authenticate()
# Initialize the library
# ee.Initialize(project='bukit30project')
# ee.Initialize(project='treeo-1')
ee.Initialize(project=name_project_gcp)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [4]:
AOIt_shp_plot = geemap.shp_to_ee(config['AOI_path'])
crs_input = config['crs_input']
I_satellite = config['I_satellite']
project_name = config['project_name']

start_date = config['date_start_end'][0]
end_date = config['date_start_end'][1]

layer_name_image_mosaick = f'image_mosaick_result_ee_{project_name}'

AOI = AOIt_shp_plot
config['AOI'] = AOI

ndwi_hi = 0.1
if config['I_satellite'] == 'Landsat':
    ndwi_hi = config['ndwi_hi_landsat']
elif I_satellite == 'Sentinel':
    ndwi_hi = config['ndwi_hi_sentinel']
elif I_satellite == 'Planet':
    ndwi_hi = config['ndwi_hi_planet']

### Masking and overlay and area helper Make an image out of the AOI area attribute -> convert featurecollection into raster (image) for overlaying tools
OID = config['OID_field_name']
AOI_img = AOI.filter(ee.Filter.notNull([OID])).reduceToImage(
    properties= [OID],
    reducer= ee.Reducer.first()
)

In [5]:
if create_training_gee == False:
    # verification format the data input content, training and AOI
    df_training_data = gpd.read_file(config['input_training'])
    print('before_validation: ',df_training_data.shape)
    # Function to check if a value is an integer
    def is_integer(value):
        return isinstance(value, int)

    # Filter out non-integer values in the 'code_lu' column
    df_training_data['code_lu'] = df_training_data['code_lu'].apply(lambda x: x if is_integer(int(x)) else None)
    display(df_training_data)
    print('after validation: (if the same, no error found)',df_training_data.shape)

elif create_training_gee == True and is_first_run == False:
    config['input_training'] = new_sample_gee_created
    # verification format the data input content, training and AOI
    df_training_data = gpd.read_file(config['input_training'])
    print('before_validation: ',df_training_data.shape)
    # Function to check if a value is an integer
    def is_integer(value):
        return isinstance(value, int)

    # Filter out non-integer values in the 'code_lu' column
    df_training_data['code_lu'] = df_training_data['code_lu'].apply(lambda x: x if is_integer(int(x)) else None)
    display(df_training_data)
    print('after validation: (if the same, no error found)',df_training_data.shape)

# Create a pandas DataFrame from the data AOI
df_AOI = gpd.read_file(config['AOI_path'])
display(df_AOI)

fields = df_AOI.columns.tolist()
print(fields)
#for area id in shapefile that identified the data, and will converted into raster
OID = config['OID_field_name']  #IMPORTANT TO CHECK OID based on the column ID
if OID not in fields:
    print(f'field_name of {OID} is not exist ERROR WILL HAPPEN!!!!')
    raise ValueError(f"Field '{OID}' not found in the fields: {fields}")
else:
    print(f"Field '{OID}' found. Proceeding with operations masking based on AOI \nplease continue")
    # Proceed with further operations, like converting to raster, etc.
    #############################################
    ##################################################################################
    ### Masking and overlay and area helper Make an image out of the AOI area attribute -> convert featurecollection into raster (image) for overlaying tools
    AOI_img = AOI.filter(ee.Filter.notNull([OID])).reduceToImage(
        properties= [OID],
        reducer= ee.Reducer.first()
    )

<class 'fiona.errors.DriverError'>: C:\Users\q_bal\Documents\GEE_notebook_Forestry\osi\00_input/training_merged_2023June_updated_2024_dec_updated.shp: No such file or directory